In [88]:
import torch
import torch.nn as nn
import pandas as pd

In [89]:
class AttentionBlock(nn.Module):
    def __init__(self, n_heads, d_model):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(4*d_model, d_model)
        )
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout = 0.15, batch_first = True)

    def forward(self, x, mask):
        x_normed = self.norm1(x)
        attention_output, _ = self.attention(x_normed,x_normed,x_normed, key_padding_mask = mask)
        x = x + attention_output
        x = x + self.mlp(self.norm2(x))
        return x

In [90]:
class Encoder(nn.Module):
    def __init__(self, n_heads, d_model, n_blocks, vocab_size, max_tokens):
        super().__init__()

        self.n_heads = n_heads
        self.d_model = d_model
        self.n_blocks = n_blocks
        self.vocab_size = vocab_size

        self.embeddings = nn.Embedding(self.vocab_size, d_model)
        self.positional_embedding = nn.Embedding(max_tokens, d_model)
        self.attention_blocks = nn.ModuleList([
            AttentionBlock(n_heads, d_model) for _ in range(self.n_blocks)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_model // 2, 4)
        )

    def forward(self, x, attention_mask):
        B, T = x.shape
        mask = (attention_mask == 0)
        positions = torch.arange(T, device = x.device)
        x = self.embeddings(x) + self.positional_embedding(positions) #(B,T)->(B,T,C)

        for block in self.attention_blocks:
            x = block(x, mask)
        x = self.norm(x)
        x = x[:,0,:]
        x = self.classifier(x)
        return x
        

In [91]:
df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

In [92]:
df_train

,id,comment_text,toxic,severe_toxic,obscene,insult
0,691941fe3255065e,Christmas Is Where Family's Come Together To C...,1,0,1,1
1,c63d9e4b8c3417bd,"Im afraid you are wrong, i was never an admin ...",0,0,0,0
2,10a596707e7a5141,The orange infobox only has the hieroglyphs an...,0,0,0,0
3,16f740ee63e59ed0,"Completed item 1, when you're done with item 2...",0,0,0,0
4,b585008394bd8a2e,Contested deletion \n\nThis article should not...,0,0,0,0
...,...,...,...,...,...,...
39995,52d63df9facc5378,I really hope you faggots know how to swim bec...,1,0,0,0
39996,c771262430e24f6e,WHAT THE FUCK IS YOUR PROBLEM!!!!!!!!!!!!!!!!!...,1,1,1,1
39997,2dd3dc26fc66828a,Come to my house right now and do me in the bu...,0,0,1,0
39998,b0bf4747b5210024,"Sock-Puppetry \n\nFor the second time, this ti...",0,0,0,0


In [93]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

In [94]:
example = tokenizer("hi im cool")

In [95]:
example

{'input_ids': [101, 7632, 10047, 4658, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

In [96]:
padding_index = tokenizer.pad_token_id
vocab_size = tokenizer.vocab_size

In [97]:
from torch.utils.data import Dataset, DataLoader

In [98]:
class SentenceDataset(Dataset):
    def __init__(self, texts, y=None):
        self.texts = texts
        self.y = y
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        text = self.texts[index]
        if self.y is not None:
            labels = self.y[index]
        else:
            labels = (0,0,0,0)

        return text, labels

In [99]:
full_data = SentenceDataset(
    df_train["comment_text"],
    [
        (x["toxic"], x["severe_toxic"], x["obscene"], x["insult"])
        for _, x in df_train.iterrows()
    ]
)

train_size = int(0.8 * len(full_data))
val_size = len(full_data) - train_size

train_data, validation_data = torch.utils.data.random_split(
    full_data,
    [train_size, val_size]
)

In [100]:
test_data = SentenceDataset(df_test['comment_text'])

In [101]:
def collate_fn(batch):
    texts = []
    labels = []
    for x,y in batch:
        texts.append(x)
        labels.append(y)

    output = tokenizer(texts, max_length=128, padding=True, truncation=True, return_tensors="pt")

    return output['input_ids'], output['attention_mask'], torch.tensor(labels)

In [102]:
train_dataloader = DataLoader(train_data, batch_size= 64, collate_fn=collate_fn, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size= 64, collate_fn=collate_fn, shuffle=False)
test_dataloader = DataLoader(test_data, batch_size= 64, collate_fn=collate_fn, shuffle=False)

In [103]:
device = "cpu" if torch.mps.is_available() == False else "mps"

In [104]:
sum(df_train['obscene'])

4355

In [105]:
model = Encoder(4,128,3,vocab_size,128)
model = model.to(device)
optimizer = torch.optim.AdamW(params=model.parameters(),lr = 1e-4, weight_decay=1e-3)
epochs = 13
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([sum(df_train['toxic']) / len(df_train), sum(df_train['severe_toxic']) / len(df_train),sum(df_train['obscene']) / len(df_train), sum(df_train['insult']) / len(df_train)], device = device))

In [106]:
from tqdm import tqdm

In [107]:
next(iter(train_dataloader))

(tensor([[  101,  4931,  1010,  ...,     0,     0,     0],
         [  101,  1000,  2839,  ...,     0,     0,     0],
         [  101,  1000,  1010,  ..., 16655, 23048,   102],
         ...,
         [  101,  1045,  2123,  ...,     0,     0,     0],
         [  101,  7592,  1010,  ...,     0,     0,     0],
         [  101,  1000,  1045,  ...,     0,     0,     0]]),
 tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 tensor([[1, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [1, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [1, 0, 0, 0],
         [0, 0, 0, 0],
         [1, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [1, 0, 1, 1],
         [0, 0, 0, 0],
         [0, 

In [108]:
x = (0,0,0,0)

In [109]:
from sklearn.metrics import accuracy_score, f1_score


for epoch in range(epochs):
    train_losses = []
    validation_losses = []
    train_predictions = []
    true_train_labels = []
    valid_predictions = []
    true_valid_labels = []
    total_train_loss = 0.
    total_validation_loss = 0.

    train_bar = tqdm(train_dataloader, desc = f"Training on epoch {epoch}...")
    valid_bar = tqdm(validation_dataloader, desc = f"Validation on epoch {epoch}...")

    model.train()
    for ids, mask, y_true in train_bar:

        ids = ids.to(device)
        mask = mask.to(device)
        y_true = y_true.to(device).float()

        optimizer.zero_grad()

        y_hat = model(ids, mask)
        train_predictions.append(torch.flatten(torch.sigmoid(y_hat) > 0.5).detach().cpu())
        true_train_labels.append(y_true.flatten().detach().cpu())

        loss = loss_fn(y_hat, y_true)

        train_losses.append(loss.item())
        total_train_loss += loss.item()
        train_bar.set_postfix(loss = loss.item())

        loss.backward()
        optimizer.step()
    train_predictions = torch.cat(train_predictions, dim=0)
    true_train_labels = torch.cat(true_train_labels, dim=0)
    avg_train_loss = total_train_loss / len(train_bar)
    print(f"Average train loss for epoch {epoch} is {avg_train_loss}")
    print(f"Accuracy of classification is {f1_score(true_train_labels, train_predictions)}")

    model.eval()
    with torch.inference_mode():
        for ids, mask, y_true in valid_bar:
        
                ids = ids.to(device)
                mask = mask.to(device)
                y_true = y_true.to(device).float()
                
                y_hat = model(ids, mask) #(B, 4)
                valid_predictions.append(torch.flatten(torch.sigmoid(y_hat) > 0.5).detach().cpu())
                true_valid_labels.append(y_true.flatten().detach().cpu())
        
                loss = loss_fn(y_hat, y_true)
        
                validation_losses.append(loss.item())
                total_validation_loss += loss.item()
                valid_bar.set_postfix(loss = loss.item())

        valid_predictions = torch.cat(valid_predictions, dim=0)
        true_valid_labels = torch.cat(true_valid_labels, dim=0)
        avg_validation_loss = total_validation_loss / len(valid_bar)
        print(f"Average valid loss for epoch {epoch} is {avg_validation_loss}")
        print(f"Accuracy of validation classification is {f1_score(true_valid_labels, valid_predictions)}")


Training on epoch 0...: 100%|██████████| 500/500 [00:22<00:00, 22.34it/s, loss=0.0581]














Average train loss for epoch 0 is 0.0999258358180523
Accuracy of classification is 0.033102122106998144









































































































































Validation on epoch 0...: 100%|██████████| 125/125 [00:24<00:00,  5.02it/s, loss=0.0514]


Average valid loss for epoch 0 is 0.052446812570095064
Accuracy of validation classification is 0.21380607037335483


Training on epoch 1...: 100%|██████████| 500/500 [00:22<00:00, 22.47it/s, loss=0.0474]


...

Average train loss for epoch 9 is 0.02862909828033298
Accuracy of classification is 0.6379719839077117


Validation on epoch 9...: 100%|██████████| 125/125 [00:29<00:00,  4.28it/s, loss=0.0285]


Average valid loss for epoch 9 is 0.03491509371250868
Accuracy of validation classification is 0.5637982195845698


Training on epoch 10...:  23%|██▎       | 117/500 [00:06<00:19, 19.48it/s, loss=0.0278]


KeyboardInterrupt: 

In [ ]:
test_bar = tqdm(test_dataloader)

  0%|          | 0/71 [00:00<?, ?it/s]

In [ ]:
model.eval()
predictions = []
with torch.inference_mode():
    for ids, mask, y_true in test_bar:
        
            ids = ids.to(device)
            mask = mask.to(device)
            y_true = y_true.to(device).float()
                
            y_hat = model(ids, mask) #(B, 4)
            
            print(torch.sigmoid(y_hat[:1, :]))
            predictions.append([(torch.sigmoid(y_hat[:,0]) > 0.95), (torch.sigmoid(y_hat[:,1]) > 0.95), (torch.sigmoid(y_hat[:,2]) > 0.1), (torch.sigmoid(y_hat[:,3]) > 0.02)])

tensor([[9.4003e-01, 4.7432e-04, 3.9377e-02, 6.0930e-02]], device='mps:0')
tensor([[0.9990, 0.0110, 0.9721, 0.7481]], device='mps:0')
tensor([[8.4049e-01, 6.3017e-05, 1.3644e-02, 1.8618e-02]], device='mps:0')
tensor([[0.9985, 0.0055, 0.9433, 0.3514]], device='mps:0')
tensor([[0.9988, 0.0188, 0.9565, 0.8608]], device='mps:0')
tensor([[1.2331e-01, 3.8093e-05, 2.3921e-03, 6.5436e-03]], device='mps:0')
tensor([[5.3782e-04, 1.9102e-07, 9.6646e-05, 5.4946e-05]], device='mps:0')
tensor([[3.5544e-04, 2.5990e-07, 9.4237e-05, 3.5170e-05]], device='mps:0')
tensor([[5.9168e-04, 2.2698e-07, 1.0094e-04, 5.2928e-05]], device='mps:0')
tensor([[0.9071, 0.0025, 0.3878, 0.1697]], device='mps:0')
tensor([[9.8947e-01, 2.8832e-04, 1.2802e-01, 7.2796e-02]], device='mps:0')
tensor([[8.7111e-01, 3.9498e-04, 2.5139e-02, 4.8709e-02]], device='mps:0')
tensor([[0.9994, 0.0075, 0.9679, 0.7190]], device='mps:0')
tensor([[0.9962, 0.0035, 0.9213, 0.1854]], device='mps:0')
tensor([[0.9987, 0.0116, 0.9596, 0.7754]], dev

In [ ]:
len(predictions[0][0])

64

In [ ]:
predictions[0][0][0]

tensor(False, device='mps:0')

In [ ]:
answer = []
for batch in range(len(predictions)):
    for i in range(len(predictions[batch][0])):
        ans = f"[{int(predictions[batch][0][i])}, {int(predictions[batch][1][i])}, {int(predictions[batch][2][i])}, {int(predictions[batch][3][i])}]"
        answer.append(ans)

In [ ]:
answer[4]

'[1, 0, 1, 0]'

In [ ]:
len(answer)

4521

In [ ]:
submission = pd.DataFrame({
    'datapointID':df_test['id'],
    'answer': answer,
    'subtaskID': [1] * len(df_test)
}).to_csv("submission.csv", index=False)